In [1039]:
import polars as pl

In [1040]:
from postgres_utils import conectar, ejecutar_consulta

conn = conectar()

In [1041]:
# Comprobamos si las tablas existen

for tabla in ["clientes", "productos", "ventas"]:

    resultado = ejecutar_consulta(
        conn,
        f'SELECT COUNT(*) FROM "{tabla}"'
    )

    print(f"{tabla}: {resultado[0][0]} registros")

clientes: 500 registros
productos: 150 registros
ventas: 10000 registros


In [1042]:
# 1. Select de pocas columnas:

resultado = ejecutar_consulta(
    conn,
    """
    SELECT
        id_cliente,
        nombre,
        ciudad
    FROM clientes
    LIMIT 10
    """
)

resultado

[(1, 'Álvaro', 'Valencia'),
 (2, 'Miguel', 'Santander'),
 (3, 'Elena', 'Alicante'),
 (4, 'María', 'Salamanca'),
 (5, 'Alejandro', 'Logroño'),
 (6, 'Daniel', 'Málaga'),
 (7, 'Claudia', 'Zaragoza'),
 (8, 'Javier', 'Alicante'),
 (9, 'Claudia', 'Gijón'),
 (10, 'David', 'Madrid')]

In [1043]:
# 2. ¿Qué clientes tienen 78 años o más?

resultado = ejecutar_consulta(
    conn,
    """
    SELECT
        id_cliente,
        nombre,
        apellidos,
        edad
    FROM clientes
    WHERE edad >= 78
    """
)

resultado

[(32, 'Sofía', 'Ortega Álvarez', 78),
 (36, 'Miguel', 'Martín Torres', 80),
 (105, 'Carlos', 'García Martín', 78),
 (143, 'Sofía', 'Sánchez Romero', 79),
 (154, 'Pablo', 'Pérez Torres', 80),
 (177, 'Álvaro', 'Torres Martín', 80),
 (210, 'Pablo', 'Sánchez López', 78),
 (218, 'Andrea', 'Sánchez Sánchez', 79),
 (245, 'Julia', 'Álvarez García', 79),
 (308, 'Carlos', 'Martínez García', 80),
 (343, 'Miguel', 'Pérez Moreno', 78),
 (349, 'Iván', 'López Álvarez', 80),
 (358, 'Álvaro', 'López Pérez', 78),
 (359, 'Paula', 'Pérez Moreno', 78),
 (388, 'Miguel', 'Sánchez Álvarez', 78),
 (401, 'Marta', 'Martín Torres', 80),
 (408, 'Carlos', 'Martín Gómez', 80),
 (423, 'Andrea', 'Ruiz Muñoz', 78),
 (430, 'Carlos', 'Álvarez Navarro', 80),
 (434, 'Daniel', 'Fernández Vázquez', 79)]

In [1044]:
# 3. ¿Cuáles son los 10 clientes de mayor edad?

resultado = ejecutar_consulta(
    conn,
    """
    SELECT
        nombre,
        apellidos,
        edad
    FROM clientes
    ORDER BY edad DESC
    LIMIT 10
    """
)

resultado

[('Carlos', 'Martín Gómez', 80),
 ('Miguel', 'Martín Torres', 80),
 ('Marta', 'Martín Torres', 80),
 ('Iván', 'López Álvarez', 80),
 ('Álvaro', 'Torres Martín', 80),
 ('Carlos', 'Álvarez Navarro', 80),
 ('Pablo', 'Pérez Torres', 80),
 ('Carlos', 'Martínez García', 80),
 ('Sofía', 'Sánchez Romero', 79),
 ('Julia', 'Álvarez García', 79)]

In [1045]:
# 4. ¿Número de clientes?

resultado = ejecutar_consulta(
    conn,
    """
    SELECT COUNT(*)
    FROM clientes
    """
)

print("Número de clientes:", resultado[0][0])

Número de clientes: 500


In [1046]:
# 5. ¿Edad media de los clientes?

resultado = ejecutar_consulta(
    conn,
    """
    SELECT AVG(edad)
    FROM clientes
    """
)

print("Edad media:", round(resultado[0][0]))

Edad media: 49


In [1047]:
# 6. ¿Edad máx y mín de los clientes?

resultado = ejecutar_consulta(
    conn,
    """
    SELECT
        MIN(edad),
        MAX(edad)
    FROM clientes
    """
)

print("Edad mínima:", resultado[0][0])
print("Edad máxima:", resultado[0][1])

Edad mínima: 18
Edad máxima: 80


In [1048]:
# 7. ¿Producto más caro?

resultado = ejecutar_consulta(
    conn,
    """
    SELECT
        id_producto,
        nombre,
        categoria,
        precio
    FROM productos
    ORDER BY precio DESC
    LIMIT 1
    """
)

resultado

[(13, 'Cerdo', 'Carnes', 18.52)]

In [1049]:
# 8. ¿Más barato?

resultado = ejecutar_consulta(
    conn,
    """
    SELECT
        id_producto,
        nombre,
        categoria,
        precio
    FROM productos
    ORDER BY precio ASC
    LIMIT 1
    """
)

resultado

[(86, 'Galletas', 'Desayuno', 0.5)]

In [1050]:
# 9. ¿Precio medio de los productos?

resultado = ejecutar_consulta(
    conn,
    """
    SELECT AVG(precio)
    FROM productos
    """
)

print("Precio medio:", round(resultado[0][0],2))

Precio medio: 6.27


In [1051]:
# 10. GROUP BY: permite agrupar registros que comparten un mismo valor.

# En este caso agrupamos los productos por categoría y contamos cuántos productos hay en cada una.

resultado = ejecutar_consulta(
    conn,
    """
    SELECT
        categoria,
        COUNT(*) AS numero_productos
    FROM productos
    GROUP BY categoria
    ORDER BY numero_productos DESC
    """
)

resultado

[('Dulces', 17),
 ('Snacks', 15),
 ('Mascotas', 13),
 ('Panadería', 11),
 ('Bebidas', 11),
 ('Bebé', 10),
 ('Frutas', 10),
 ('Pescados', 9),
 ('Verduras', 9),
 ('Conservas', 8),
 ('Higiene', 8),
 ('Limpieza', 7),
 ('Carnes', 6),
 ('Desayuno', 6),
 ('Congelados', 5),
 ('Lácteos', 5)]

In [1052]:
# 11. JOIN: Tenemos información repartida entre dos tablas.

# ventas contiene id_producto.

# productos contiene el nombre del producto.

# Con JOIN podemos combinar ambas tablas utilizando id_producto como campo común.

# con FROM ventas v estamos diciendo que a ventas le llamamos v en esta consulta, es un alias

resultado = ejecutar_consulta(
    conn,
    """
    SELECT
        p.id_producto,
        p.nombre,
        SUM(v.cantidad) AS unidades_vendidas
    FROM ventas v 
    JOIN productos p
        ON v.id_producto = p.id_producto
    GROUP BY
        p.id_producto,
        p.nombre
    ORDER BY unidades_vendidas DESC
    LIMIT 10
    """
)

resultado

[(112, 'Aquarius', 214),
 (16, 'Vino', 206),
 (59, 'Caramelos', 194),
 (81, 'Vino', 193),
 (150, 'Cerveza', 190),
 (23, 'Lechuga', 189),
 (147, 'Cacao', 188),
 (104, 'Pan Integral', 186),
 (48, 'Frutos secos', 185),
 (22, 'Gominolas', 181)]

In [1053]:
# 12. Pasamos la consulta a polars:

resultado = ejecutar_consulta(
    conn,
    """
    SELECT
        p.nombre,
        SUM(v.cantidad) AS unidades_vendidas
    FROM ventas v
    JOIN productos p
        ON v.id_producto = p.id_producto
    GROUP BY
        p.id_producto,
        p.nombre
    ORDER BY unidades_vendidas DESC
    """
)

df_productos = pl.DataFrame(
    resultado,
    schema=[
        "producto",
        "unidades_vendidas"
    ],
    orient="row"
)

df_productos

producto,unidades_vendidas
str,i64
"""Aquarius""",214
"""Vino""",206
"""Caramelos""",194
"""Vino""",193
"""Cerveza""",190
…,…
"""Calamares""",112
"""Brócoli""",110
"""Pan Integral""",109


In [1054]:
df_productos.head(10)

producto,unidades_vendidas
str,i64
"""Aquarius""",214
"""Vino""",206
"""Caramelos""",194
"""Vino""",193
"""Cerveza""",190
"""Lechuga""",189
"""Cacao""",188
"""Pan Integral""",186
"""Frutos secos""",185


In [1055]:
# Ejercicios
## Consultas básicas

### Ejercicio 1: ¿Cuántos clientes hay?
num_clientes = ejecutar_consulta(
    conn,
    """
    SELECT
        COUNT(id_cliente)
    FROM clientes
    """
)
num_clientes


[(500,)]

In [1056]:
### Ejercicio 2: ¿Cuántos productos hay?
num_productos = ejecutar_consulta(
    conn,
    """
    SELECT
        DISTINCT (nombre)
    FROM productos
    """
)
num_productos



[('Pienso perro',),
 ('Cereales',),
 ('Salchichas',),
 ('Pepino',),
 ('Bacalao',),
 ('Vino',),
 ('Mantequilla',),
 ('Lejía',),
 ('Pañales',),
 ('Salmón',),
 ('Pan Integral',),
 ('Lavavajillas',),
 ('Champú',),
 ('Chicles',),
 ('Zanahoria',),
 ('Agua',),
 ('Papel higiénico',),
 ('Pienso gato',),
 ('Café',),
 ('Brócoli',),
 ('Galletas',),
 ('Doritos',),
 ('Pizza',),
 ('Fanta',),
 ('Lechuga',),
 ('Cebolla',),
 ('Queso',),
 ('Aceitunas',),
 ('Cepillo dientes',),
 ('Pan de semillas',),
 ('Suavizante',),
 ('Arena gato',),
 ('Langostinos',),
 ('Nata',),
 ('Gominolas',),
 ('Helado',),
 ('Limpiacristales',),
 ('Barra',),
 ('Kiwi',),
 ('Molde',),
 ('Hamburguesa',),
 ('Zumo',),
 ('Tomate triturado',),
 ('Papilla',),
 ('Cacao',),
 ('Batido',),
 ('Mandarina',),
 ('Verduras congeladas',),
 ('Palomitas',),
 ('Atún',),
 ('Uva',),
 ('Plátano',),
 ('Frutos secos',),
 ('Naranja',),
 ('Caramelos',),
 ('Calamares',),
 ('Patatas fritas',),
 ('Aquarius',),
 ('Detergente',),
 ('Snacks perro',),
 ('Pasta de di

In [1057]:
### Ejercicio 3: ¿Cuántas ventas hay?
num_ventas = ejecutar_consulta(
    conn,
    """
   SELECT column_name
    FROM information_schema.columns
    WHERE table_name = 'productos';
    """
)
num_ventas



[('id_producto',),
 ('precio',),
 ('coste',),
 ('nombre',),
 ('categoria',),
 ('marca',)]

In [1058]:
### Ejercicio 3: ¿Cuántas ventas hay?
num_ventass = ejecutar_consulta(
    conn,
    """
    SELECT
        COUNT(id_venta)
    FROM ventas
    """
)
num_ventass



[(10000,)]

In [1059]:
### Ejercicio 4: ¿Cuál es la edad media de los clientes?
edad_media = ejecutar_consulta(
    conn,
    """
    SELECT
        avg(edad)
    FROM clientes
    """
)
edad_media



[(Decimal('49.3460000000000000'),)]

In [1060]:
### Ejercicio 5: ¿Cuál es el producto más caro?
producto_caro = ejecutar_consulta(
    conn,
    """
    SELECT
        max(precio)
    FROM PRODUCTOS
    """
)
producto_caro

[(18.52,)]

In [1061]:
### Ejercicio 6: ¿Cuál es el producto más barato?
producto_caro = ejecutar_consulta(
    conn,
    """
    SELECT
        min(precio)
    FROM PRODUCTOS
    """
)
producto_caro

[(0.5,)]

In [1062]:
### Ejercicio 7: ¿Cuántos clientes hay por provincia?
agrupacion_provincia = ejecutar_consulta(
    conn,
    """
    SELECT
        provincia,
        count(id_cliente)
    FROM clientes
    group by provincia
    """
)
agrupacion_provincia

[('Navarra', 25),
 ('Madrid', 24),
 ('Barcelona', 24),
 ('Murcia', 30),
 ('Zaragoza', 24),
 ('A Coruña', 24),
 ('Pontevedra', 27),
 ('Valencia', 26),
 ('Málaga', 28),
 ('Salamanca', 34),
 ('Alicante', 27),
 ('Bizkaia', 32),
 ('Sevilla', 23),
 ('La Rioja', 31),
 ('León', 33),
 ('Asturias', 33),
 ('Valladolid', 10),
 ('Cantabria', 27),
 ('Burgos', 18)]

In [1063]:
### Ejercicio 8: ¿Cuántos clientes hay por tipo de cliente?
agrupacion_tipoCliente = ejecutar_consulta(
    conn,
    """
    SELECT
        tipo_cliente,
        count(id_cliente)
    FROM clientes
    group by tipo_cliente
    """
)
agrupacion_tipoCliente


[('Ocasional', 106), ('Frecuente', 114), ('Normal', 280)]

In [1064]:
### Ejercicio 9: ¿Cuántos productos hay por categoría?
agrupacion_productoCategoria = ejecutar_consulta(
    conn,
    """
    SELECT
        nombre,
        count (categoria)
    FROM productos
    group by nombre,categoria
    """
)
agrupacion_productoCategoria

[('Leche infantil', 3),
 ('Ternera', 1),
 ('Salmón', 1),
 ('Turrón', 5),
 ('Cebolla', 1),
 ('Kiwi', 1),
 ('Zanahoria', 1),
 ('Uva', 2),
 ('Gominolas', 6),
 ('Lejía', 1),
 ('Molde', 3),
 ('Champú', 1),
 ('Cerveza', 2),
 ('Helado', 1),
 ('Cacao', 1),
 ('Nata', 1),
 ('Caramelos', 2),
 ('Pipas', 4),
 ('Pienso gato', 3),
 ('Batido', 1),
 ('Verduras congeladas', 1),
 ('Cerdo', 1),
 ('Salchichas', 3),
 ('Pienso perro', 3),
 ('Aquarius', 1),
 ('Brócoli', 3),
 ('Pasta de dientes', 1),
 ('Zumo', 4),
 ('Naranja', 1),
 ('Snacks perro', 5),
 ('Patatas fritas', 1),
 ('Atún', 1),
 ('Queso', 2),
 ('Merluza', 1),
 ('Maíz', 2),
 ('Langostinos', 1),
 ('Fanta', 1),
 ('Pepino', 1),
 ('Café', 1),
 ('Lechuga', 1),
 ('Pizza', 1),
 ('Aceitunas', 1),
 ('Doritos', 5),
 ('Calamares', 3),
 ('Hamburguesa', 1),
 ('Atún', 2),
 ('Barra', 1),
 ('Agua', 1),
 ('Plátano', 1),
 ('Papilla', 1),
 ('Pan Integral', 4),
 ('Chicles', 4),
 ('Lasaña', 1),
 ('Galletas', 3),
 ('Suavizante', 1),
 ('Lavavajillas', 1),
 ('Pan de semill

In [1065]:
### Ejercicio 10: ¿Cuántas ventas hay por método de pago?
agrupacion_ventasMetodo = ejecutar_consulta(
    conn,
    """
    SELECT
        metodo_pago,
        count (id_venta)
    FROM ventas
    group by metodo_pago
    """
)
agrupacion_ventasMetodo

[('Efectivo', 1495),
 ('Tarjeta', 6026),
 ('Bizum', 2005),
 ('Cheque regalo', 474)]

In [1066]:
### Ejercicio 11: ¿Cuántas unidades se han vendido de cada producto?
agrupacion_ventasProducto = ejecutar_consulta(
    conn,
    """
    SELECT
        nombre,
        COUNT(DISTINCT nombre) AS total
    FROM productos
    GROUP BY nombre
    """
)
agrupacion_ventasProducto


[('Aceitunas', 1),
 ('Agua', 1),
 ('Aquarius', 1),
 ('Arena gato', 1),
 ('Atún', 1),
 ('Bacalao', 1),
 ('Baguette', 1),
 ('Barra', 1),
 ('Batido', 1),
 ('Brócoli', 1),
 ('Cacao', 1),
 ('Café', 1),
 ('Calamares', 1),
 ('Caramelos', 1),
 ('Cebolla', 1),
 ('Cepillo dientes', 1),
 ('Cerdo', 1),
 ('Cereales', 1),
 ('Cerveza', 1),
 ('Champú', 1),
 ('Chicles', 1),
 ('Detergente', 1),
 ('Doritos', 1),
 ('Fanta', 1),
 ('Frutos secos', 1),
 ('Galletas', 1),
 ('Gominolas', 1),
 ('Hamburguesa', 1),
 ('Helado', 1),
 ('Kiwi', 1),
 ('Langostinos', 1),
 ('Lasaña', 1),
 ('Lavavajillas', 1),
 ('Leche infantil', 1),
 ('Lechuga', 1),
 ('Lejía', 1),
 ('Limpiacristales', 1),
 ('Maíz', 1),
 ('Mandarina', 1),
 ('Mantequilla', 1),
 ('Merluza', 1),
 ('Molde', 1),
 ('Naranja', 1),
 ('Nata', 1),
 ('Palomitas', 1),
 ('Pañales', 1),
 ('Pan de semillas', 1),
 ('Pan Integral', 1),
 ('Papel higiénico', 1),
 ('Papilla', 1),
 ('Pasta de dientes', 1),
 ('Patata', 1),
 ('Patatas fritas', 1),
 ('Pepino', 1),
 ('Pera', 1),


In [ ]:
## JOIN

### Ejercicio 12: Muestra cada venta junto con el nombre del cliente.
venta_cliente = ejecutar_consulta(
    conn,
    """
    SELECT
        v.id_venta,
        v.fecha,
        c.nombre,
        c.apellidos,
        v.cantidad,
        v.metodo_pago
    FROM ventas v
    JOIN clientes c
        ON v.id_cliente = c.id_cliente
    ORDER BY v.id_venta
    LIMIT 10
    """
)

venta_cliente


In [ ]:
### Ejercicio 13: Muestra cada venta junto con el nombre del producto.
venta_producto = ejecutar_consulta(
    conn,
    """
    SELECT
        v.id_venta,
        p.nombre AS producto,
        v.cantidad,
        p.precio,
        v.metodo_pago
    FROM ventas v
    JOIN productos p
        ON v.id_producto = p.id_producto
    ORDER BY v.id_venta
    LIMIT 10
    """
)

venta_producto


In [ ]:
### Ejercicio 14: Muestra cada venta con cliente, producto, cantidad y método de pago.
ventas_detalle = ejecutar_consulta(
    conn,
    """
    SELECT
        c.nombre AS cliente,
        c.apellidos,
        p.nombre AS producto,
        v.cantidad,
        v.metodo_pago
    FROM ventas v
    JOIN clientes c
        ON v.id_cliente = c.id_cliente
    JOIN productos p
        ON v.id_producto = p.id_producto
    ORDER BY v.id_venta
    """
)

ventas_detalle


In [ ]:
## Extra

### Ejercicio 15: ¿Cuáles son los 10 clientes con más ventas?
top_clientes = ejecutar_consulta(
    conn,
    """
    SELECT
        c.id_cliente,
        c.nombre,
        c.apellidos,
        COUNT(v.id_venta) AS numero_ventas
    FROM clientes c
    JOIN ventas v
        ON c.id_cliente = v.id_cliente
    GROUP BY
        c.id_cliente,
        c.nombre,
        c.apellidos
    ORDER BY numero_ventas DESC
    LIMIT 10
    """
)

top_clientes


In [ ]:
### Ejercicio 16: ¿Cuáles son los 10 productos con más unidades vendidas?
top_productos = ejecutar_consulta(
    conn,
    """
    SELECT
        p.id_producto,
        p.nombre,
        SUM(v.cantidad) AS unidades_vendidas
    FROM productos p
    JOIN ventas v
        ON p.id_producto = v.id_producto
    GROUP BY
        p.id_producto,
        p.nombre
    ORDER BY unidades_vendidas DESC
    LIMIT 10
    """
)

top_productos


In [ ]:
### Ejercicio 17: ¿Qué método de pago se utiliza más?
metodo_pago_top = ejecutar_consulta(
    conn,
    """
    SELECT
        metodo_pago,
        COUNT(*) AS numero_ventas
    FROM ventas
    GROUP BY metodo_pago
    ORDER BY numero_ventas DESC
    LIMIT 1
    """
)

metodo_pago_top


In [ ]:
### Ejercicio 18: ¿Qué provincia tiene más clientes?
provincia_top = ejecutar_consulta(
    conn,
    """
    SELECT
        provincia,
        COUNT(*) AS numero_clientes
    FROM clientes
    GROUP BY provincia
    ORDER BY numero_clientes DESC
    LIMIT 1
    """
)

provincia_top


In [ ]:
### Ejercicio 19: ¿Qué categoría tiene más productos?
categoria_top = ejecutar_consulta(
    conn,
    """
    SELECT
        categoria,
        COUNT(*) AS numero_productos
    FROM productos
    GROUP BY categoria
    ORDER BY numero_productos DESC
    LIMIT 1
    """
)

categoria_top


In [ ]:
### Ejercicio 20: ¿Qué categoría ha generado más ingresos por ventas?
ingresos_categoria = ejecutar_consulta(
    conn,
    """
    SELECT
        p.categoria,
        SUM(v.cantidad * p.precio) AS ingresos
    FROM ventas v
    JOIN productos p
        ON v.id_producto = p.id_producto
    GROUP BY p.categoria
    ORDER BY ingresos DESC
    LIMIT 1
    """
)

ingresos_categoria


In [1068]:
conn.close()